# Setpoint Feedforward for an Ore Feed Conveyor

Setpoint feedforward control of the ore feed rate delivered by a long
conveyor belt, simulated with the python-control library. Adapted from the
GEL-7063 exercise `ex_anticipation_consigne.slx` (anticipation de la
consigne).

The control system is a 2-DOF structure with model-based setpoint
feedforward:

![Setpoint feedforward control system](images/setpoint_ff_diagram.png)

$$
U(s) = G_{ff}(s) R(s) + G_c(s) \left( G_r(s) R(s) - Y(s) \right)
$$

$$
Y(s) = G_p(s) U(s) + D(s)
$$

where:
 - $R(s)$ : setpoint
 - $G_p(s)$ : plant (reclaim feeder and conveyor)
 - $G_m(s)$ : model of the plant, used to design $G_{ff}(s)$ and $G_r(s)$
 - $G_c(s)$ : PI feedback controller
 - $G_{ff}(s)$ : setpoint feedforward
 - $G_r(s)$ : reference model (desired setpoint response)
 - $D(s)$ : output disturbance

The feedforward and the reference model are designed from the model of
the plant and a reference filter $F_r(s)$, which specifies the desired
setpoint tracking dynamics (user choice). The model is decomposed into an
invertible part $\tilde{G}_m(s)$ and a non-invertible part
$\bar{G}_m(s)$ (time delays and right-half-plane zeros):

$$
G_m(s) = \tilde{G}_m(s) \bar{G}_m(s)
$$

$$
G_{ff}(s) = \frac{F_r(s)}{\tilde{G}_m(s)}
$$

$$
G_r(s) = F_r(s) \bar{G}_m(s)
$$

In [ ]:
from pathlib import Path

import control as con
import matplotlib.pyplot as plt
import numpy as np

plot_dir = Path('plots')
plot_dir.mkdir(exist_ok=True)

## Plant

The conveyor belt is loaded by a variable speed reclaim feeder. To allow
for the momentum of the feeder, its response to the speed setpoint $u$ is
modelled as a second-order lag with time constant $T_p = 3$ s. The conveyor is $L = 90$ m long and runs at
a fixed speed of $v = 1$ m/s, so ore loaded onto the belt reaches the
discharge after a transport delay of $\theta = L / v = 90$ s. The output
$y$ is the ore feed rate at the conveyor discharge:

$$
G_p(s) = \frac{K_p}{(T_p s + 1)^2} e^{-\theta s}
= \frac{1}{(3s + 1)^2} e^{-90s}
$$

Signals are in normalized units, so the gain is $K_p = 1$.

python-control transfer functions are rational, so the time delay is
kept separate from the rational part $G_{p,0}(s)$. For simulation, the
models are converted to discrete time, where the delay is represented
exactly as a shift of $\theta / T_s$ samples (see
[Discretization](#Discretization)).

In [ ]:
s = con.tf('s')

# Plant parameters
K_p = 1.0      # gain (normalized units)
T_p = 3.0      # plant time constant (s), second-order lag
L = 90.0       # conveyor length (m)
v = 1.0        # conveyor speed (m/s)
theta = L / v  # transport delay (s)

# Rational part of plant
G_p0 = K_p / (T_p * s + 1) ** 2
print(G_p0)

In [ ]:
# Step response of the plant (the rational part shifted by the delay)
t_step = np.linspace(0, 150, 1501)
y_step = con.step_response(G_p0, T=t_step).outputs
y_step = np.interp(t_step - theta, t_step, y_step, left=0.0)

fig, ax = plt.subplots(figsize=(7, 2.5))
ax.plot(t_step, y_step)
ax.set_xlabel('Time (s)')
ax.set_ylabel('$y(t)$')
ax.set_title('Plant step response')
ax.grid()
plt.tight_layout()
plt.savefig(plot_dir / 'conveyor_plant_step.png', dpi=150)
plt.show()

## Discretization

All blocks are converted to discrete time with a zero-order hold (ZOH)
at sample time $T_s$. The time delay is then an exact shift of
$n_d = \theta / T_s$ samples:

$$
G_p(z) = G_{p,0}(z) \, z^{-n_d}
$$

The other blocks are discretized in the same way where they are defined
below.

The ZOH conversion is exact for blocks whose input is constant between
samples (e.g. the setpoint path when $r(t)$ steps at a sample time). In
the feedback loop the results are an approximation of the continuous
system, with errors that decrease with $T_s$ (less than 0.01 in $y(t)$ for
$T_s = 0.5$ s).

In [ ]:
# Sample time
Ts = 0.5

# Time delay as a shift of n_d samples
n_d = round(theta / Ts)
assert np.isclose(n_d * Ts, theta), "delay must be a multiple of Ts"
z_delay = con.ss(con.tf([1], [1] + [0] * n_d, Ts))

# Discrete-time plant (ZOH)
G_p_d = z_delay * con.ss(con.c2d(G_p0, Ts, 'zoh'))
print(f"Delay: {n_d} samples")

## Closed-Loop System

The function below constructs the closed-loop system from its component
transfer functions. Each block is given named input and output signals,
and the blocks are connected with `con.interconnect`
(see the block diagram in its docstring).

Only the feedback controller $G_c$ and the plant $G_p$ are required. The
other blocks are optional:

| Argument | Block | Default |
|---|---|---|
| `G_r` | Reference model, $r \rightarrow r_m$ | 1 |
| `G_ff` | Setpoint feedforward, $r \rightarrow u_{ff}$ | `None` (no feedforward) |

The disturbance $d$ is added directly to the plant output.

In Simulation 2, $G_r$ is the reference filter $F_r$ alone. In the
simulations with setpoint feedforward, it is the reference model
$G_r = F_r \bar{G}_m$.

In [ ]:
def make_closed_loop_system(G_c, G_p, G_r=1, G_ff=None, name='closed_loop'):
    """Construct a closed-loop system with optional setpoint feedforward.

    Block diagram:

                 ┌──────┐ u_ff
            ┌───►│ G_ff │─────────────────────────┐
            │    └──────┘                         │                  d
            │                                     │                  │
            │   ┌─────┐ r_m      e  ┌─────┐ u_fb  ▼+ u ┌─────┐ y_p   ▼+
        r ──┴──►│ G_r │──────►(Σ)──►│ G_c │─────►(Σ)──►│ G_p │─────►(Σ)──┬──► y
                └─────┘      + ▲−   └─────┘     +      └─────┘     +     │
                               │                                         │
                               └─────────────────────────────────────────┘

    All systems must have the same sample time as G_p (or all be
    continuous-time). Scalars are treated as static gains.

    Args:
        G_c: Feedback controller, e -> u_fb.
        G_p: Plant, u -> y_p.
        G_r: Reference model, r -> r_m (default 1).
        G_ff: Setpoint feedforward, r -> u_ff (default None, no
            feedforward).
        name: Name of the closed-loop system.

    Returns:
        Closed-loop system with inputs (r, d) and outputs
        (r_m, e, u, y), where d is an output disturbance.
    """
    dt = G_p.dt

    def block(sys, inputs, outputs, name):
        if np.isscalar(sys):
            sys = con.tf(sys, 1, dt)
        return con.ss(sys, inputs=inputs, outputs=outputs, name=name)

    blocks = [
        block(G_r, 'r', 'r_m', 'G_r'),
        block(G_c, 'e', 'u_fb', 'G_c'),
        block(G_p, 'u', 'y_p', 'G_p'),
        con.summing_junction(
            inputs=['r_m', '-y'], output='e', dt=dt, name='sum_e'
        ),
        con.summing_junction(
            inputs=['y_p', 'd'], output='y', dt=dt, name='sum_y'
        ),
    ]
    if G_ff is None:
        u_inputs = ['u_fb']
    else:
        blocks.append(block(G_ff, 'r', 'u_ff', 'G_ff'))
        u_inputs = ['u_ff', 'u_fb']
    blocks.append(
        con.summing_junction(inputs=u_inputs, output='u', dt=dt, name='sum_u')
    )

    return con.interconnect(
        blocks, inputs=['r', 'd'], outputs=['r_m', 'e', 'u', 'y'], name=name
    )

## Simulation

Simulation parameters:

| Parameter | Value |
|---|---|
| Simulation time | 0 – 1200 s |
| Setpoint step $r(t)$ | 0 → 1 at $t = 10$ s |
| Disturbance step $d(t)$ | 0 → 1 at $t = 600$ s |
| Initial conditions | zero |

The discrete-time closed-loop systems are simulated at the sample times
$t = k T_s$.

In [ ]:
# Simulation parameters
t_stop = 1200.0
t_r, r_step = 10.0, 1.0     # setpoint step time and size
t_d, d_step = 600.0, 1.0    # disturbance step time and size

t = Ts * np.arange(round(t_stop / Ts) + 1)
r = np.where(t >= t_r - Ts / 2, r_step, 0.0)
d = np.where(t >= t_d - Ts / 2, d_step, 0.0)


def simulate(T):
    """Simulate closed-loop system T with the inputs (r, d).

    Returns:
        Dictionary of output signals (r_m, e, u, y).
    """
    sim = con.forced_response(T, T=t, U=[r, d])
    return {name: sim.outputs[T.find_output(name)] for name in T.output_labels}

In [ ]:
def plot_sim_results(t, r, sim, title=None, filename=None):
    """Plot the setpoint, output, error and control signals, and save
    the figure to plot_dir if a filename is given."""
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(7, 1 + 3*1.5))

    ax = axes[0]
    ax.plot(t, sim['y'], label='$y(t)$')
    ax.plot(t, r, label='$r(t)$')
    ax.plot(t, sim['r_m'], '--', label='$r_m(t)$')
    ax.set_ylabel('$r(t)$, $y(t)$')
    ax.set_title(title)
    ax.legend(loc='lower right', fontsize='small')

    ax = axes[1]
    ax.plot(t, sim['e'])
    ax.set_ylabel('$e(t)$')

    ax = axes[2]
    ax.plot(t, sim['u'])
    ax.set_ylabel('$u(t)$')
    ax.set_xlabel('Time (s)')

    for ax in axes:
        ax.grid()
    plt.tight_layout()
    if filename is not None:
        plt.savefig(plot_dir / filename, dpi=150)
    plt.show()

## PI Feedback Controller

$$
G_c(s) = K_c \frac{T_i s + 1}{T_i s}
$$

The controller is tuned with the improved SIMC rule (Skogestad and
Grimholt, 2012). The plant is first approximated by a first-order plus
time delay model $k e^{-\theta' s} / (\tau_1 s + 1)$ using Skogestad's half
rule: half of the second time constant is added to the first time
constant and half to the time delay,

$$
\tau_1 = T_p + \frac{T_p}{2}, \qquad \theta' = \theta + \frac{T_p}{2}
$$

The improved rule replaces $\tau_1$ by $\tau_1 + \theta'/3$ in the SIMC PI
rule, which gives better performance for delay-dominant processes:

$$
K_c = \frac{1}{k} \frac{\tau_1 + \theta'/3}{\tau_c + \theta'}, \qquad
T_i = \min \left( \tau_1 + \theta'/3, \; 4(\tau_c + \theta') \right)
$$

The closed-loop time constant $\tau_c$ is the tuning parameter. The
recommended choice $\tau_c = \theta'$ gives a robust, smooth response.

Reference: S. Skogestad and C. Grimholt (2012), "The SIMC method for
smooth PID controller tuning", in *PID Control in the Third Millennium*,
Springer.

In [ ]:
# First-order plus time delay approximation (half rule)
tau_1 = T_p + T_p / 2      # time constant (s)
theta_1 = theta + T_p / 2  # time delay (s)

# Improved SIMC tuning
tau_c = theta_1  # closed-loop time constant (s)
K_c = (tau_1 + theta_1 / 3) / (K_p * (tau_c + theta_1))
T_i = min(tau_1 + theta_1 / 3, 4 * (tau_c + theta_1))
print(f"K_c = {K_c:.4f}, T_i = {T_i:.1f} s")

G_c = K_c * (T_i * s + 1) / (T_i * s)
G_c_d = con.c2d(G_c, Ts, 'zoh')

### Simulation 1: Feedback Control Only

The closed-loop system with the PI controller only: no reference filter
($G_r = 1$) and no setpoint feedforward.

In [ ]:
T_fb = make_closed_loop_system(G_c_d, G_p_d, name='feedback_only')
sim_fb = simulate(T_fb)
plot_sim_results(t, r, sim_fb, title='Feedback control only',
                 filename='conveyor_feedback_only.png')

## Reference Filter

The reference filter $F_r(s)$ specifies the desired setpoint tracking
dynamics (user choice). Since the invertible part of the plant model is
second order, a second-order filter is needed to make the feedforward
proper:

$$
F_r(s) = \frac{1}{(T_F s + 1)^2}
$$

In [ ]:
# Reference filter parameters
T_F = 10.0     # filter time constant (s)

F_r = 1 / (T_F * s + 1) ** 2
F_r_d = con.c2d(F_r, Ts, 'zoh')
print(F_r)

### Simulation 2: Reference Filter

The setpoint is filtered by $F_r(s)$ before the error is computed
($G_r = F_r$), with no setpoint feedforward.

In [ ]:
T_filt = make_closed_loop_system(G_c_d, G_p_d, G_r=F_r_d, name='ref_filter')
sim_filt = simulate(T_filt)
plot_sim_results(t, r, sim_filt, title='Reference filter',
                 filename='conveyor_ref_filter.png')

## Setpoint Feedforward and Reference Model

The setpoint feedforward is designed from the reference filter $F_r(s)$
and a model of the plant.

### Plant Model Decomposition

The model of the plant is assumed to be perfect, $G_m(s) = G_p(s)$. It
is decomposed into invertible and non-invertible parts:

$$
G_m(s) = \tilde{G}_m(s) \bar{G}_m(s)
= \frac{1}{(3s + 1)^2} \cdot e^{-90s}
$$

The non-invertible part $\bar{G}_m(s)$ is the transport delay. As before,
the delay is kept separate from the rational part, which here is
$\bar{G}_{m,0}(s) = 1$.

In [ ]:
# Model of the plant (perfect model, rational part)
G_m0 = G_p0

# Invertible part
G_m_inv = K_p / (T_p * s + 1) ** 2

# Non-invertible part (rational part)
G_m_ninv0 = con.tf([1], [1])

# Check that the two parts recombine to the model rational part
diff = con.minreal(G_m_inv * G_m_ninv0 - G_m0, verbose=False)
assert np.allclose(diff.num[0][0], 0.0)
print(G_m_inv)
print(G_m_ninv0)

### Feedforward and Reference Model

The feedforward controller and the resulting tracking dynamics are:

$$
G_{ff}(s) = \frac{F_r(s)}{\tilde{G}_m(s)} = \frac{(3s + 1)^2}{(10s + 1)^2}
$$

$$
G_r(s) = F_r(s) \bar{G}_m(s) = \frac{1}{(10s + 1)^2} e^{-90s}
$$

$G_r(s)$ is the reference model: the setpoint response obtained with the
feedforward alone when the model is perfect, $G_r(s) = G_{ff}(s) G_m(s)$.
The feedback controller acts on the error between this reference
trajectory and the plant output.

In [ ]:
# Feedforward (minreal cancels any common factors)
G_ff = con.minreal(F_r / G_m_inv, verbose=False)
print(G_ff)

In [ ]:
# Reference model (rational part)
G_r0 = F_r * G_m_ninv0
print(G_r0)

In [ ]:
# Check that G_ff * G_m0 equals the reference model rational part
diff = con.minreal(G_ff * G_m0 - G_r0, verbose=False)
assert np.allclose(diff.num[0][0], 0.0)
print(con.minreal(G_ff * G_m0, verbose=False))

### Discretization of the Feedforward

The ZOH discretization of a product of transfer functions is not the
product of their ZOH discretizations. If $G_{ff}(s)$ and $G_r(s)$ were
discretized independently, $G_{ff}(z) G_m(z)$ would differ slightly from
$G_r(z)$ and the error $e(t)$ would be small but non-zero even with a
perfect model.

Forming the discrete feedforward as $F_r(z) / \tilde{G}_m(z)$ avoids this
but is not a good option here: the ZOH discretization of the second-order
$\tilde{G}_m(s)$ has a sampling zero on the negative real axis (near
$z = -0.89$), which would become a pole of the feedforward and cause an
oscillating feedforward signal.

Instead, $G_{ff}(s)$ is discretized directly, and the discrete reference
model is defined as the feedforward path through the discrete model of
the plant:

$$
G_r(z) = G_{ff}(z) G_m(z)
$$

so that the reference trajectory is exactly the output of the model when
driven by the feedforward. It is very close to the ZOH discretization of
the continuous $G_r(s)$.

In [ ]:
# Discrete-time feedforward (ZOH)
G_ff_d = con.c2d(G_ff, Ts, 'zoh')

# Discrete-time model of the plant and reference model
G_m_d = z_delay * con.ss(con.c2d(G_m0, Ts, 'zoh'))
G_r_d = G_m_d * con.ss(G_ff_d)

# Compare with the ZOH discretization of the continuous reference model
t_check = Ts * np.arange(round(200 / Ts) + 1)
y_r = con.step_response(G_r_d, T=t_check).outputs
y_F = con.step_response(con.ss(con.c2d(G_r0, Ts, 'zoh')) * z_delay,
                        T=t_check).outputs
print(G_ff_d)
print(f"Max difference from c2d(G_r): {np.abs(y_r - y_F).max():.4f}")

### Simulation 3: Setpoint Feedforward

The setpoint is passed through the reference model $G_r$, with the
setpoint feedforward $G_{ff}$.

In [ ]:
T_ff = make_closed_loop_system(
    G_c_d, G_p_d, G_r=G_r_d, G_ff=G_ff_d, name='setpoint_ff'
)
sim_ff = simulate(T_ff)
plot_sim_results(t, r, sim_ff, title='Setpoint feedforward',
                 filename='conveyor_setpoint_ff.png')

### Simulation 4: Plant-Model Mismatch in the Time Delay

The feedforward $G_{ff}$ and reference model $G_r$ are designed with the
model delay $\theta = 90$ s. Here the actual plant delay $\theta_p$ is
shorter (70 s) or longer (110 s) than the model delay, e.g. due to a change
in conveyor speed. The feedforward then no longer produces the reference
trajectory exactly, so the error $e(t)$ is non-zero during the setpoint
change and the feedback controller has to act.

In [ ]:
def make_discrete_plant(theta_p):
    """Discrete-time plant (ZOH) with the rational part G_p0 and time
    delay theta_p (must be a multiple of Ts)."""
    n = round(theta_p / Ts)
    assert np.isclose(n * Ts, theta_p), "delay must be a multiple of Ts"
    delay = con.ss(con.tf([1], [1] + [0] * n, Ts))
    return delay * con.ss(con.c2d(G_p0, Ts, 'zoh'))


# Actual plant delays (s)
plant_delays = [70.0, theta, 110.0]

sims_mm = {}
for theta_p in plant_delays:
    T_mm = make_closed_loop_system(
        G_c_d, make_discrete_plant(theta_p), G_r=G_r_d, G_ff=G_ff_d,
        name='setpoint_ff_mismatch'
    )
    sims_mm[rf'$\theta_p = {theta_p:g}$ s'] = simulate(T_mm)

In [ ]:
def plot_sim_comparison(t, r, sims, title=None, filename=None):
    """Plot the output, error and control signals of several
    simulations, given as a dictionary {label: sim}, and save the
    figure to plot_dir if a filename is given."""
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(7, 1 + 3*1.5))

    ax = axes[0]
    ax.plot(t, r, 'k-', lw=1, label='$r(t)$')
    for label, sim in sims.items():
        ax.plot(t, sim['y'], label=label)
    ax.plot(t, sim['r_m'], 'k--', lw=1, label='$r_m(t)$')
    ax.set_ylabel('$r(t)$, $y(t)$')
    ax.set_title(title)
    ax.legend(loc='lower right', fontsize='small')

    for label, sim in sims.items():
        axes[1].plot(t, sim['e'], label=label)
        axes[2].plot(t, sim['u'], label=label)
    axes[1].set_ylabel('$e(t)$')
    axes[2].set_ylabel('$u(t)$')
    axes[2].set_xlabel('Time (s)')

    for ax in axes:
        ax.grid()
    plt.tight_layout()
    if filename is not None:
        plt.savefig(plot_dir / filename, dpi=150)
    plt.show()


plot_sim_comparison(t, r, sims_mm, title='Plant-model mismatch in the time delay',
                    filename='conveyor_delay_mismatch.png')

### Simulation 5: Plant-Model Mismatch with SIMC PID Control

Simulation 4 is repeated with a PID feedback controller tuned with the
SIMC rules for a second-order plus time delay process (Skogestad, 2003):

$$
g(s) = \frac{k}{(\tau_1 s + 1)(\tau_2 s + 1)} e^{-\theta s}
$$

$$
K_c = \frac{1}{k} \frac{\tau_1}{\tau_c + \theta}, \qquad
\tau_I = \min \left\{ \tau_1, \; 4(\tau_c + \theta) \right\}, \qquad
\tau_D = \tau_2
$$

These settings are for the series (interactive) form of the PID
controller, in which the PI and PD parts are multiplied.
A first-order filter with time constant $\alpha \tau_D$ is added to the
derivative term to make the controller proper:

$$
G_c(s) = K_c \frac{\tau_I s + 1}{\tau_I s} \cdot
\frac{\tau_D s + 1}{\alpha \tau_D s + 1}
$$

For the feeder, $\tau_1 = \tau_2 = T_p = 3$ s and $\theta = 90$ s, and
$\tau_c = \theta$ is used as before. The controller then cancels both
plant poles. The setpoint feedforward and reference model are unchanged.

Reference: S. Skogestad (2003), "Simple analytic rules for model reduction
and PID controller tuning", *Journal of Process Control*, 13, 291–309.

In [ ]:
# SIMC PID tuning (series form)
tau_c_pid = theta  # closed-loop time constant (s)
K_c_pid = T_p / (K_p * (tau_c_pid + theta))
tau_I = min(T_p, 4 * (tau_c_pid + theta))
tau_D = T_p
alpha = 0.1        # derivative filter factor
print(f"K_c = {K_c_pid:.4f}, tau_I = {tau_I:.1f} s, tau_D = {tau_D:.1f} s")

G_c_pid = (
    K_c_pid * (tau_I * s + 1) / (tau_I * s)
    * (tau_D * s + 1) / (alpha * tau_D * s + 1)
)
G_c_pid_d = con.c2d(G_c_pid, Ts, 'zoh')
print(G_c_pid)

In [ ]:
sims_mm_pid = {}
for theta_p in plant_delays:
    T_mm = make_closed_loop_system(
        G_c_pid_d, make_discrete_plant(theta_p), G_r=G_r_d, G_ff=G_ff_d,
        name='setpoint_ff_mismatch_pid'
    )
    sims_mm_pid[rf'$\theta_p = {theta_p:g}$ s'] = simulate(T_mm)

plot_sim_comparison(t, r, sims_mm_pid,
                    title='Plant-model mismatch in the time delay, SIMC PID',
                    filename='conveyor_delay_mismatch_pid.png')

With the PID controller, there is a small derivative kick in $u(t)$ when
the output disturbance occurs at $t = 600$ s, limited by the derivative
filter.

On a near-pure-delay process like this, derivative action buys almost
nothing. That is consistent with the finding of Grimholt and Skogestad
(2018) that for pure time delay processes the optimal PID controller is a
PI controller.

Reference: C. Grimholt and S. Skogestad (2018), "Should we forget the
Smith Predictor?", *IFAC-PapersOnLine*, 51(4), 769–774.

## Accounting for Plant Uncertainty

Devasia (2002) asks when an inversion-based feedforward improves tracking
compared to feedback alone, when the plant differs from the model. The
control structure in the paper is

$$
U(s) = G_m^{-1}(s) Y_d(s) + G_c(s) \left( Y_d(s) - Y(s) \right)
$$

This is the structure used above, with the desired output
$Y_d(s) = G_r(s) R(s)$ and $G_{ff}(s) = G_m^{-1}(s) G_r(s) = F_r(s) / \tilde{G}_m(s)$
(the delay in $G_m^{-1}$ cancels the delay in $G_r$).

Let the plant uncertainty be the difference between the model and the
plant (the model $G_m$ is the paper's nominal plant $G_0$):

$$
\Delta(s) = G_m(s) - G_p(s)
$$

The tracking errors $E = Y_d - Y$ with and without the feedforward are
(Eqs. 6 and 7):

$$
E_{ff}(j\omega) = \left[ 1 + G_p G_c \right]^{-1} \Delta \, G_m^{-1} Y_d(j\omega),
\qquad
E_{fb}(j\omega) = \left[ 1 + G_p G_c \right]^{-1} Y_d(j\omega)
$$

so that, in the SISO case,

$$
|E_{ff}(j\omega)| = \left| \frac{\Delta(j\omega)}{G_m(j\omega)} \right| |E_{fb}(j\omega)|
$$

**Uncertainty acceptability condition** (Condition 2, Lemma 1). For a MIMO
system, the feedforward does not make the worst-case tracking error worse,
for any feedback controller, at frequencies where

$$
\| \Delta(j\omega) \|_2 \le \frac{\| G_m(j\omega) \|_2}{\kappa_{G_m}(j\omega)},
\qquad
\kappa_{G_m}(j\omega) = \| G_m(j\omega) \|_2 \, \| G_m^{-1}(j\omega) \|_2
$$

where $\| \cdot \|_2$ is the induced 2-norm (maximum singular value) and
$\kappa_{G_m}$ is the condition number of the model.

**SISO case** (Theorem 1). The condition number is 1, and the condition
is necessary and sufficient:

$$
|\Delta(j\omega)| \le |G_m(j\omega)|
\quad \Longleftrightarrow \quad
\left| 1 - \frac{G_p(j\omega)}{G_m(j\omega)} \right| \le 1
$$

i.e. the relative model error must be at most 100% at that frequency. When
it holds, the feedforward improves tracking for every desired output, and
independently of the feedback controller $G_c$ (Remark 5).

**Modified inverse** (Definition 2). Since most models have large
uncertainty in some frequency range (usually at high frequencies), the
inverse is only applied where the condition holds:

$$
G_{ff\Delta}(j\omega) =
\begin{cases}
G_m^{-1}(j\omega) & \text{if } \| \Delta(j\omega) \|_2 \le
\| G_m(j\omega) \|_2 / \kappa_{G_m}(j\omega) \\
0 & \text{otherwise}
\end{cases}
$$

With this feedforward, tracking is improved compared to feedback alone at
every frequency, for any feedback controller. The modified inverse is a
special case (Lemma 7) of the optimal inverse that minimizes a
frequency-weighted cost on the input and the tracking error (Lemma 6):

$$
U_{opt}(j\omega) = \left[ R(j\omega) + G_m^*(j\omega) Q(j\omega) G_m(j\omega) \right]^{-1}
G_m^*(j\omega) Q(j\omega) Y_d(j\omega)
$$

with $R = 0, Q = I$ where the condition holds and $R = I, Q = 0$ where it
does not. Note that an ideal cut-off in frequency makes the modified
inverse noncausal (Lemma 5, Paley–Wiener), so a causal implementation
must approximate it, e.g. with a low-pass filter.

Reference: S. Devasia (2002), "Should model-based inverse inputs be used
as feedforward under plant uncertainty?", *IEEE Transactions on Automatic
Control*, 47(11), 1865–1871.

### Uncertainty Bound for a Time Delay Mismatch

For the delay mismatch in Simulation 4, the plant is the model with a
different delay, $G_p(s) = G_m(s) e^{-\delta\theta s}$ with
$\delta\theta = \theta_p - \theta$. The relative uncertainty is

$$
\left| \frac{\Delta(j\omega)}{G_m(j\omega)} \right|
= \left| 1 - e^{-j \omega \delta\theta} \right|
= 2 \left| \sin \left( \frac{\omega \delta\theta}{2} \right) \right|
$$

which is at most 1 when $|\sin(\omega \delta\theta / 2)| \le 1/2$. In the
lowest frequency band, the feedforward improves tracking for

$$
\omega \le \omega_{max} = \frac{\pi}{3 |\delta\theta|}
$$

The condition is met again in bands centred on multiples of
$2\pi / |\delta\theta|$, but in practice the feedforward would only be
used below $\omega_{max}$.

In [ ]:
# Delay mismatch (s); the relative uncertainty depends only on its size
delta_theta = max(abs(p - theta) for p in plant_delays)

w = np.logspace(-3, 0, 1000)  # frequency (rad/s)
rel_unc = np.abs(1 - np.exp(-1j * w * delta_theta))
F_r_mag = np.abs(1 / (T_F * 1j * w + 1) ** 2)

# Upper limit of the lowest frequency band where the condition holds
w_max = np.pi / (3 * delta_theta)

fig, ax = plt.subplots(figsize=(7, 3))
ax.loglog(w, rel_unc, label=rf'$|\Delta / G_m|$, $|\delta\theta| = {delta_theta:g}$ s')
ax.loglog(w, F_r_mag, '--', label=r'$|F_r(j\omega)|$')
ax.axhline(1, color='k', lw=1)
ax.axvline(w_max, color='k', ls=':', lw=1)
ax.set_xlabel('Frequency (rad/s)')
ax.set_ylabel('Magnitude')
ax.set_title('Uncertainty acceptability condition (Devasia, 2002)')
ax.set_ylim(1e-2, 3)
ax.legend(loc='lower left', fontsize='small')
ax.grid(which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(plot_dir / 'conveyor_ff_uncertainty_bound.png', dpi=150)
plt.show()

F_r_w_max = np.abs(1 / (T_F * 1j * w_max + 1) ** 2)
print(f"w_max = {w_max:.4f} rad/s, |F_r(j w_max)| = {F_r_w_max:.3f}")

For $|\delta\theta| = 20$ s, the condition holds for
$\omega \le \omega_{max} \approx 0.052$ rad/s. The reference filter with
$T_F = 10$ s has a bandwidth of about 0.064 rad/s, and its gain is
still about 0.78 at $\omega_{max}$, so the desired output contains frequencies
where the feedforward makes tracking worse. This suggests two design
options for the feedforward under delay uncertainty:

1. Slow down the reference filter, so that $Y_d$ has little content above
   $\omega_{max}$.
2. Keep $F_r(s)$ and approximate the modified inverse by adding a low-pass
   filter to $G_{ff}(s)$ with a cut-off near $\omega_{max}$.

### Simulation 6: Slower Reference Filter

The reference filter cannot change the frequencies where the feedforward
improves tracking, but it can move the content of the desired output
$Y_d = G_r R$ below $\omega_{max}$. Here the filter is slowed down so that
its corner frequency $1/T_F$ is at the edge of the band where the
condition holds for both delay mismatches:

$$
T_F = \frac{1}{\omega_{max}} = \frac{3 \max |\delta\theta|}{\pi}
$$

For $\max |\delta\theta| = 20$ s, this gives $T_F = 19.1$ s, rounded up to
20 s. The filter gain at $\omega_{max}$ is then 0.5, compared to 0.78 with
$T_F = 10$ s. The feedforward and reference model are redesigned with the
new filter in the same way as before, and Simulation 4 is repeated.

In [ ]:
# Slower reference filter with its corner frequency at w_max
T_F_slow = np.ceil(1 / w_max)  # filter time constant (s)
print(f"1 / w_max = {1 / w_max:.1f} s, T_F_slow = {T_F_slow:g} s")

F_r_slow = 1 / (T_F_slow * s + 1) ** 2
G_ff_slow = con.minreal(F_r_slow / G_m_inv, verbose=False)
G_ff_slow_d = con.c2d(G_ff_slow, Ts, 'zoh')
G_r_slow_d = G_m_d * con.ss(G_ff_slow_d)
print(G_ff_slow)

In [ ]:
sims_mm_slow = {}
for theta_p in plant_delays:
    T_mm = make_closed_loop_system(
        G_c_d, make_discrete_plant(theta_p), G_r=G_r_slow_d, G_ff=G_ff_slow_d,
        name='setpoint_ff_mismatch_slow'
    )
    sims_mm_slow[rf'$\theta_p = {theta_p:g}$ s'] = simulate(T_mm)

plot_sim_comparison(
    t, r, sims_mm_slow,
    title=f'Plant-model mismatch in the time delay, $T_F = {T_F_slow:g}$ s',
    filename='conveyor_delay_mismatch_slow_filter.png'
)

In [ ]:
# Integrated absolute errors during the setpoint change (before the
# disturbance): e = r_m - y measures tracking of the reference trajectory,
# r - y also includes the slower trajectory itself.
k = t < t_d
print(f"{'':12}{'IAE of e(t)':^24}{'IAE of r(t) - y(t)':^24}")
cols = [f'T_F = {T_F:g} s', f'T_F = {T_F_slow:g} s']
print(f"{'theta_p (s)':>12}" + ''.join(f"{c:>12}" for c in 2 * cols))
for theta_p, label in zip(plant_delays, sims_mm):
    sims = (sims_mm[label], sims_mm_slow[label])
    iae_e = [Ts * np.sum(np.abs(sim['e'][k])) for sim in sims]
    iae_r = [Ts * np.sum(np.abs(r[k] - sim['y'][k])) for sim in sims]
    print(f"{theta_p:12g}" + ''.join(f"{x:12.1f}" for x in iae_e + iae_r))

With the slower filter, the peak tracking error $|e(t)|$ during the
setpoint change falls from 0.63 to 0.35 for both delay mismatches, and the
error is smoother. The IAE of $e(t)$ falls less (by about 10% for
$\theta_p = 70$ s and 3% for $\theta_p = 110$ s), because most of the
error comes from the low-frequency content of the setpoint change, where
the feedforward already improves tracking.

The cost is a slower setpoint response: the IAE of $r(t) - y(t)$
increases by about 20 for all three plants, including the nominal case.
Slowing the filter further would reduce the error caused by the delay
mismatch, but slows the response to every setpoint change, whether or not
there is a mismatch. The modified inverse (option 2) would instead keep
the faster reference trajectory and only cut the feedforward above
$\omega_{max}$.

## Pulse-Step Setpoint Feedforward (Visioli, 2004)

Visioli (2004) proposes a nonlinear feedforward for setpoint step changes.
The inverse-model feedforward used above (the "classic" scheme in the
paper, $G_{ff} = G_r / \tilde{G}_m$, Eq. 1) is replaced by a
piecewise-constant signal. It drives the model output from its initial
value to the new setpoint in a specified transition time $\tau$, and then
holds it there. The feedback controller is tuned for load disturbance
rejection, and acts only on the error between the reference trajectory and
the plant output, as before.

The notation of the paper is translated as follows:

| Paper | Here | Description |
|---|---|---|
| $y_{sp}$ | $r$ | setpoint |
| $y_f$ | $r_m$ | reference trajectory (input to the feedback loop) |
| $M(s)$ | $G_r(s)$ | reference model (classic scheme) |
| $G(s)$ | $G_{ff}(s)$ | feedforward (classic scheme) |
| $P(s)$ | $G_m(s)$ | model of the plant |
| $\tilde{P}(s)$ | $\tilde{G}_m(s)$ | minimum-phase part of the model |
| $F(s)$ | $F_{ps}(s)$ | reference filter (pulse-step scheme) |
| $C(s)$ | $G_c(s)$ | feedback controller |

### Feedforward Signal

The plant is described by a first-order plus time delay (FOPDT) model
(Eq. 2):

$$
G_m(s) = \frac{K}{T s + 1} e^{-L s}
$$

For a setpoint step from $0$ to $y_1 > 0$ at $t = 0$, the feedforward
signal is (Eq. 3)

$$
u_{ff}(t) =
\begin{cases}
\bar{u}_{ff} & \text{if } 0 \le t < \tau \\
y_1 / K & \text{if } t \ge \tau
\end{cases}
$$

The model output is zero until $t = L$, and then rises as
$K \bar{u}_{ff} \left( 1 - e^{-(t - L)/T} \right)$. The pulse amplitude
$\bar{u}_{ff}$ is chosen so that the model output reaches $y_1$ at
$t = \tau + L$ (Eq. 4):

$$
\bar{u}_{ff} = \frac{y_1 / K}{1 - e^{-\tau / T}}
$$

At $t = \tau$ the input switches to its steady-state value $y_1 / K$.
Since the FOPDT model has a single state, its output then stays at $y_1$
from $t = \tau + L$. With a perfect model, the output transition takes
place in the interval $[L, \tau + L]$.

### Reference Trajectory

The reference trajectory $r_m$ for the feedback loop is the output of the
model driven by $u_{ff}$. It is obtained by filtering the setpoint step
with (Eq. 5)

$$
F_{ps}(s) = \frac{K \bar{u}_{ff} / y_1}{T s + 1} e^{-L s}
$$

and saturating the result at $y_1$:

$$
r_m(t) =
\begin{cases}
0 & \text{if } t < L \\
\min \left( K \bar{u}_{ff} \left( 1 - e^{-(t - L)/T} \right), \; y_1 \right)
& \text{if } t \ge L
\end{cases}
$$

so that $r_m = G_m u_{ff}$. With a perfect model, $e(t) = 0$ and the
feedback controller output stays at zero during the setpoint change.

**Setpoint changes during a transition** (Remark 3). The feedforward
signal and reference trajectory for a new setpoint change are computed in
the same way for the size of the change, and added to those of the
previous changes.

**Choice of the transition time** (Remarks 2 and 4). A shorter $\tau$ gives
a larger pulse $\bar{u}_{ff}$, so $\tau$ trades off speed of response
against control effort and robustness. Alternatively, $\bar{u}_{ff}$ can be
chosen first (e.g. as a fraction of the actuator limit), and Eq. 4
rearranged to give the transition time:

$$
\tau = T \ln \left( \frac{\bar{u}_{ff}}{\bar{u}_{ff} - y_1 / K} \right)
$$

### Model Uncertainty

Since $r_m = G_m u_{ff}$, the tracking error with a plant $G_p$ that
differs from the model is

$$
E(s) = \frac{G_m(s) - G_p(s)}{1 + G_p(s) G_c(s)} U_{ff}(s)
= \frac{\Delta(s)}{1 + G_p(s) G_c(s)} U_{ff}(s)
$$

i.e. the modelling error acts like a load disturbance driven by
$u_{ff}$, which the feedback controller has to reject. The paper states
this in the same way, following Wallén (2000), and recommends tuning the
feedback controller for load disturbance rejection. It does not address
when the feedforward improves tracking under uncertainty, and refers to
Devasia (2002) for this.

Compared with the inverse-model feedforward, the pulse-step feedforward
gives a faster transition for the same peak control signal, because it
holds the input at its maximum value for the whole transition. In the
paper's examples, the robustness to modelling errors was about the same
for both schemes, and their responses became more similar as the
transition time increased.

References:
- A. Visioli (2004), "A new design for a PID plus feedforward
  controller", *Journal of Process Control*, 14, 457–463.
- A. Wallén and K. J. Åström (2002), "Pulse-step control", *Preprints of
  the 15th IFAC World Congress*, Barcelona.

### Application to the Conveyor

The method assumes a FOPDT model, but the model of the conveyor is second
order plus time delay. The pulse-step feedforward is therefore designed
from the FOPDT approximation of the model given by the half rule (see
[PI Feedback Controller](#PI-Feedback-Controller)), $K = K_p = 1$ and
$T = \tau_1 = 4.5$ s. Only $K$ and $T$ are needed to compute $u_{ff}(t)$
from Eqs. 3 and 4.

The reference trajectory is not computed from the FOPDT model (Eq. 5).
Instead, it is the output of the full discrete model of the plant driven by
the feedforward signal:

$$
r_m(t) = G_m(z) \, u_{ff}(t)
$$

This is the same approach used for the inverse-model feedforward, where
$G_r(z) = G_{ff}(z) G_m(z)$. With a perfect model the error $e(t)$ is
zero, so the feedback controller does not react to the error of the FOPDT
approximation. The cost is that the second-order model has two states, so
its output does not stop exactly at the setpoint when the input switches to
its steady-state value at $t = \tau$. The reference trajectory overshoots
slightly before it settles. Removing this would need a pulse with two
levels, as in the pulse-step control of Wallén and Åström (2002).

The transition time is chosen to be close to the time constant of the
FOPDT approximation, $\tau = 5$ s (a multiple of $T_s$, so that the
switching is at a sample time and the ZOH discretization is exact). The
pulse amplitude is then $\bar{u}_{ff} \approx 1.49$, compared with a peak
of 1 for the inverse-model feedforward of Simulation 3.

In [ ]:
def pulse_step_feedforward(t, t_step, step, K, T, tau):
    """Pulse-step feedforward signal (Visioli, 2004, Eqs. 3 and 4) for
    a setpoint step of size step at time t_step, designed from a FOPDT
    model with gain K and time constant T, and transition time tau."""
    u_bar = (step / K) / (1 - np.exp(-tau / T))
    u_ff = np.where(t >= t_step - Ts / 2, step / K, 0.0)
    pulse = (t >= t_step - Ts / 2) & (t < t_step + tau - Ts / 2)
    return np.where(pulse, u_bar, u_ff)


# FOPDT approximation of the model (half rule) and transition time
K_fo = K_p
T_fo = tau_1
tau_ps = 5.0  # transition time (s)
assert np.isclose(round(tau_ps / Ts) * Ts, tau_ps), "tau must be a multiple of Ts"

u_ff_ps = pulse_step_feedforward(t, t_r, r_step, K_fo, T_fo, tau_ps)

# Reference trajectory: output of the full model driven by u_ff
r_m_ps = con.forced_response(G_m_d, T=t, U=u_ff_ps).outputs

print(f"u_bar = {u_ff_ps.max():.3f}")
print(f"Peak of r_m: {r_m_ps.max():.4f}")

### Closed-Loop System with Time-Series Inputs

The function `make_closed_loop_system` generates the reference trajectory
$r_m$ and the feedforward $u_{ff}$ from the setpoint $r$ with transfer
functions. Here they are time series, so the variant below takes them as
external inputs. The rest of the system is linear, so the simulation is
still exact. Setpoint changes during a transition (Remark 3) can be
simulated by adding the signals for each change.

In [ ]:
def make_closed_loop_system_ext(G_c, G_p, name='closed_loop_ext'):
    """Construct a closed-loop system with external reference trajectory
    and feedforward inputs.

    Block diagram:

                  u_ff ────────────┐                  d
                                   │                  │
                   e ┌─────┐ u_fb  ▼+ u ┌─────┐ y_p   ▼+
        r_m ──►(Σ)──►│ G_c │─────►(Σ)──►│ G_p │─────►(Σ)──┬──► y
              + ▲−   └─────┘     +      └─────┘     +     │
                │                                         │
                └─────────────────────────────────────────┘

    All systems must have the same sample time.

    Args:
        G_c: Feedback controller, e -> u_fb.
        G_p: Plant, u -> y_p.
        name: Name of the closed-loop system.

    Returns:
        Closed-loop system with inputs (r_m, u_ff, d) and outputs
        (e, u, y), where d is an output disturbance.
    """
    dt = G_p.dt
    blocks = [
        con.ss(G_c, inputs='e', outputs='u_fb', name='G_c'),
        con.ss(G_p, inputs='u', outputs='y_p', name='G_p'),
        con.summing_junction(
            inputs=['r_m', '-y'], output='e', dt=dt, name='sum_e'
        ),
        con.summing_junction(
            inputs=['u_ff', 'u_fb'], output='u', dt=dt, name='sum_u'
        ),
        con.summing_junction(
            inputs=['y_p', 'd'], output='y', dt=dt, name='sum_y'
        ),
    ]
    return con.interconnect(
        blocks, inputs=['r_m', 'u_ff', 'd'],
        outputs=['e', 'u', 'y'], name=name
    )


def simulate_ext(T, r_m, u_ff):
    """Simulate closed-loop system T with the inputs (r_m, u_ff, d).

    Returns:
        Dictionary of signals (r_m, e, u, y), including the input r_m
        for plotting.
    """
    sim = con.forced_response(T, T=t, U=[r_m, u_ff, d])
    outputs = {name: sim.outputs[T.find_output(name)]
               for name in T.output_labels}
    return {'r_m': r_m, **outputs}

### Simulation 7: Pulse-Step Feedforward

The pulse-step feedforward with the nominal plant and the PI controller of
Simulation 3.

In [ ]:
T_ps = make_closed_loop_system_ext(G_c_d, G_p_d, name='pulse_step')
sim_ps = simulate_ext(T_ps, r_m_ps, u_ff_ps)
plot_sim_results(t, r, sim_ps, title='Pulse-step feedforward',
                 filename='conveyor_pulse_step.png')

### Simulation 8: Pulse-Step Feedforward with Plant-Model Mismatch

Simulation 4 is repeated with the pulse-step feedforward. The feedforward
signal and the reference trajectory are designed for the model delay
$\theta = 90$ s, and the actual plant delay is 70, 90 or 110 s.

In [ ]:
sims_mm_ps = {}
for theta_p in plant_delays:
    T_mm = make_closed_loop_system_ext(
        G_c_d, make_discrete_plant(theta_p), name='pulse_step_mismatch'
    )
    sims_mm_ps[rf'$\theta_p = {theta_p:g}$ s'] = simulate_ext(
        T_mm, r_m_ps, u_ff_ps
    )

plot_sim_comparison(
    t, r, sims_mm_ps,
    title='Plant-model mismatch in the time delay, pulse-step feedforward',
    filename='conveyor_delay_mismatch_pulse_step.png'
)

In [ ]:
# Comparison with the inverse-model feedforward (Simulation 4) during
# the setpoint change (before the disturbance)
k = t < t_d
schemes = {'inverse': sims_mm, 'pulse-step': sims_mm_ps}
print(f"{'':12}{'peak |e(t)|':^24}{'IAE of e(t)':^24}{'IAE of r(t) - y(t)':^24}")
print(f"{'theta_p (s)':>12}" + ''.join(f"{c:>12}" for c in 3 * list(schemes)))
for theta_p, label in zip(plant_delays, sims_mm):
    sims = [s[label] for s in schemes.values()]
    peak_e = [np.abs(sim['e'][k]).max() for sim in sims]
    iae_e = [Ts * np.sum(np.abs(sim['e'][k])) for sim in sims]
    iae_r = [Ts * np.sum(np.abs(r[k] - sim['y'][k])) for sim in sims]
    print(f"{theta_p:12g}" + ''.join(f"{x:12.2f}" for x in peak_e + iae_e + iae_r))
print(f"\nPeak u(t): inverse {sim_ff['u'][k].max():.2f}, "
      f"pulse-step {sim_ps['u'][k].max():.2f}")

With the nominal plant (Simulation 7), the error $e(t)$ is zero and the
feedback controller is inactive during the setpoint change, as with the
inverse-model feedforward. The output reaches 95% of the setpoint change
97.5 s after the step, compared with 138 s in Simulation 3, and overshoots
by 2.2% because of the second lag of the plant. The pulse raises the peak
of $u(t)$ to 1.49, compared with 1.00 for the inverse-model feedforward.

With a delay mismatch (Simulation 8), the faster transition is a
disadvantage. The reference trajectory and the output are both close to
steps, offset in time by $\delta\theta = \pm 20$ s, so the peak error is
about the full setpoint change (1.02, compared with 0.63 for the
inverse-model feedforward). The feedback controller reacts to this short,
large error and causes an overshoot to 1.26 for $\theta_p = 110$ s
(1.16 with the inverse-model feedforward) and a dip to 0.74 for
$\theta_p = 70$ s (0.84). The IAE of $e(t)$ is about the same for the two
schemes, and the IAE of $r(t) - y(t)$ is about 16 lower with the
pulse-step feedforward for all three plants, because of the faster
transition.

This agrees with the error equation above,
$E = \Delta S \, U_{ff}$: a larger, earlier feedforward signal produces a
larger error when the model is wrong. A longer transition time $\tau$
would reduce the peak error under mismatch, at the cost of a slower
response, which is the same trade-off as the speed of the reference filter
$F_r(s)$ for the inverse-model feedforward.